In [29]:
import pandas as pd

df = pd.read_csv("flights_sample_10000.csv")
df.head()


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,DAY,DEP_HOUR,DELAYED
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,153.0,1065.0,NaN,NaN,NaN,NaN,NaN,Wednesday,11,0
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,189.0,1399.0,NaN,NaN,NaN,NaN,NaN,Saturday,21,0
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,87.0,680.0,NaN,NaN,NaN,NaN,NaN,Friday,9,0
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,249.0,1589.0,0.0,0.0,24.0,0.0,0.0,Monday,16,0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,153.0,985.0,NaN,NaN,NaN,NaN,NaN,Sunday,18,0


In [30]:
df = df[df["CANCELLED"] == 0]
df = df[df["DIVERTED"] == 0]


In [31]:
# Ubah tanggal ke format datetime
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

# Ambil nama hari (Monday, dst)
df["DAY"] = df["FL_DATE"].dt.day_name()

# Ambil jam dari CRS_DEP_TIME (misal 1345 → 13)
df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100

# Target: delay atau tidak
df["DELAYED"] = (df["DEP_DELAY"] > 15).astype(int)

df[["DAY", "DEP_HOUR", "DELAYED"]].head()


,DAY,DEP_HOUR,DELAYED
0,Wednesday,11,0
1,Saturday,21,0
2,Friday,9,0
3,Monday,16,0
4,Sunday,18,0


In [32]:
X = df[
    ["DEP_HOUR", "DISTANCE", "AIRLINE_CODE", "ORIGIN", "DEST", "DAY"]
]

y = df["DELAYED"]


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numerical),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Rows:", len(df), "| Delay rate:", round(y.mean(), 3))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

Rows: 9730 | Delay rate: 0.171
              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1612
           1       0.33      0.01      0.02       334

    accuracy                           0.83      1946
   macro avg       0.58      0.50      0.46      1946
weighted avg       0.74      0.83      0.75      1946

ROC-AUC: 0.6309926672709173


In [34]:
categorical = ["AIRLINE_CODE", "ORIGIN", "DEST", "DAY"]
numerical = ["DEP_HOUR", "DISTANCE"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", "passthrough", numerical)
    ]
)

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)


/home/everytee08/miniconda3/envs/ta-env-linux/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['AIRLINE_CODE', 'ORIGIN',
                                                   'DEST', 'DAY']),
                                                 ('num', 'passthrough',
                                                  ['DEP_HOUR', 'DISTANCE'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [35]:
model.fit(X, y)


/home/everytee08/miniconda3/envs/ta-env-linux/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['AIRLINE_CODE', 'ORIGIN',
                                                   'DEST', 'DAY']),
                                                 ('num', 'passthrough',
                                                  ['DEP_HOUR', 'DISTANCE'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [36]:
df.groupby("AIRLINE_CODE")["DELAYED"].mean().sort_values(ascending=False)

AIRLINE_CODE
B6    0.291925
F9    0.225962
G4    0.216867
WN    0.212766
EV    0.208333
NK    0.200608
AA    0.191011
OH    0.190083
UA    0.185233
DL    0.145985
OO    0.133880
YX    0.122661
YV    0.119565
MQ    0.116667
QX    0.116667
AS    0.110429
9E    0.104859
HA    0.087302
Name: DELAYED, dtype: float64

In [37]:
import joblib

joblib.dump(model, "D:\\project\\flight_dashboard\\model\\delay_model.pkl")


['D:\\project\\flight_dashboard\\model\\delay_model.pkl']

In [38]:
sample = pd.DataFrame({
    "DEP_HOUR": [17],
    "DISTANCE": [800],
    "AIRLINE_CODE": ["AA"],
    "ORIGIN": ["JFK"],
    "DEST": ["LAX"],
    "DAY": ["Friday"]
})

model.predict(sample)
model.predict_proba(sample)


array([[0.40588035, 0.59411965]])

In [39]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("Accuracy :", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_test, y_pred) * 100, 2), "%")
print("Recall   :", round(recall_score(y_test, y_pred) * 100, 2), "%")
print("F1-Score :", round(f1_score(y_test, y_pred) * 100, 2), "%")
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 4))

              precision    recall  f1-score   support

           0       0.91      0.66      0.77      1612
           1       0.30      0.69      0.42       334

    accuracy                           0.67      1946
   macro avg       0.60      0.68      0.59      1946
weighted avg       0.81      0.67      0.71      1946

Accuracy : 66.55 %
Precision: 29.71 %
Recall   : 69.46 %
F1-Score : 41.61 %
ROC-AUC  : 0.7345


In [40]:
df.to_csv("D:\\project\\flight_dashboard\\data\\flights_sample_10000.csv", index=False)


In [41]:
for col in ["AIRLINE_CODE", "DAY", "DEP_HOUR"]:
    print(df.groupby(col)["DELAYED"].agg(["mean", "count"])
            .sort_values("mean", ascending=False).head(5), "\n")

                  mean  count
AIRLINE_CODE                 
B6            0.291925    322
F9            0.225962    208
G4            0.216867    166
WN            0.212766   1880
EV            0.208333     48 

              mean  count
DAY                      
Friday    0.182442   1458
Sunday    0.180843   1399
Thursday  0.179363   1444
Monday    0.171237   1495
Saturday  0.165869   1254 

              mean  count
DEP_HOUR                 
2         0.500000      2
19        0.268000    500
21        0.264407    295
18        0.249147    586
20        0.235160    438 



In [42]:
print("Rows:", len(df))
print("Overall delay rate:", round(df["DELAYED"].mean() * 100, 1), "%")
print("Before noon:", round(df[df["DEP_HOUR"] < 12]["DELAYED"].mean() * 100, 1), "%")
print("6-9 PM     :", round(df[df["DEP_HOUR"].between(18, 21)]["DELAYED"].mean() * 100, 1), "%")

Rows: 9730
Overall delay rate: 17.1 %
Before noon: 10.8 %
6-9 PM     : 25.3 %
